In [25]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import sys

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

JSON_ROOT = Path("/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/CT json")
REWEIGHTING_ROOT = Path("/Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/reweighting")

In [ ]:
df = pd.read_csv(data_root / 'CBOS_data_to_weight_corrected.csv', low_memory=False)
print(f"Loaded: {df.shape[0]:,} obs × {df.shape[1]} cols, {df['survey_file'].nunique()} surveys")

df['teryt_id_VOIV'] = df['teryt_id_VOIV'].astype(str).str.zfill(7)

# Assign macroregions to the dataframe
cbos_macroregions = {1: ['1400000', '1000000'],
                     2: ['1200000', '2400000'],
                     3: ['0600000', '1800000', '2000000', '2600000'],
                     4: ['3000000', '3200000', '0800000'],
                     5: ['0200000', '1600000'],
                     6: ['2200000', '2800000', '0400000']
                     }

df['macroregion'] = df['teryt_id_VOIV'].apply(
    lambda x: next((int(mr) for mr, voivs in cbos_macroregions.items() if x in voivs), np.nan)
)

# Create household-level weight columns (start as copies of individual weights)
df['weight_VOIV_h']          = df['weight_VOIV']
df['weight_VOIV_500_h']      = df['weight_VOIV_500']
df['weight_VOIV_100_500_h']  = df['weight_VOIV_100_500']
df['weight_VOIV_100_h']      = df['weight_VOIV_100']

# Initialize columns for macroregion-level weights (start as copies of individual weights)
df['weight_MACRO']           = df['weight_VOIV']
df['weight_MACRO_500']       = df['weight_VOIV_500']
df['weight_MACRO_100_500']   = df['weight_VOIV_100_500']
df['weight_MACRO_100']       = df['weight_VOIV_100']

df['weight_MACRO_h']         = df['weight_VOIV_h']
df['weight_MACRO_500_h']     = df['weight_VOIV_500_h']
df['weight_MACRO_100_500_h'] = df['weight_VOIV_100_500_h']
df['weight_MACRO_100_h']     = df['weight_VOIV_100_h']

print(f"Weight columns initialised. Shape: {df.shape}")
print("Years:", sorted(df['survey_year'].unique()))

for idx, row in df.iterrows():
    if row['survey_year'] == 2017 and row['survey_month'] > 2:
        df.at[idx, 'macroregion'] = row['location_new']

Loaded: 355,337 obs × 91 cols, 327 surveys
Weight columns initialised. Shape: (355337, 96)
Years: [np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]


In [3]:
df

,Unnamed: 0,org_id,survey_file,survey_year,survey_month,age,year_born,sex,sex_L,city_size,...,weight_VOIV_100_500,weight_VOIV_100,G_VOIV_500,G_VOIV_100_500,G_VOIV_100,macroregion,weight_VOIV_h,weight_VOIV_500_h,weight_VOIV_100_500_h,weight_VOIV_100_h
0,0,1.0,CBOS_1_01_1990.sav,1990,1,33.0,1957.0,2.0,Kobieta,9.0,...,1.000000,1.000000,Group 106,Group 106,Group 106,NaN,1.000000,1.000000,1.000000,1.000000
1,1,2.0,CBOS_1_01_1990.sav,1990,1,40.0,1950.0,2.0,Kobieta,9.0,...,1.000000,1.000000,Group 106,Group 106,Group 106,NaN,1.000000,1.000000,1.000000,1.000000
2,2,3.0,CBOS_1_01_1990.sav,1990,1,58.0,1932.0,2.0,Kobieta,9.0,...,1.000000,1.000000,Group 9,Group 11,Group 11,NaN,1.000000,1.000000,1.000000,1.000000
3,3,4.0,CBOS_1_01_1990.sav,1990,1,44.0,1946.0,1.0,Mężczyzna,9.0,...,1.000000,1.000000,Group 9,Group 11,Group 11,NaN,1.000000,1.000000,1.000000,1.000000
4,7,8.0,CBOS_1_01_1990.sav,1990,1,64.0,1926.0,2.0,Kobieta,9.0,...,1.000000,1.000000,Group 3,Group 5,Group 5,NaN,1.000000,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355332,358520,921.0,CBOS_331_12_2017.sav,2017,12,60.0,1957.0,1.0,Mężczyzna,1.0,...,0.611965,0.611965,Group 69,Group 70,Group 70,NaN,0.611965,0.611965,0.611965,0.611965
355333,358521,922.0,CBOS_331_12_2017.sav,2017,12,36.0,1981.0,1.0,Mężczyzna,1.0,...,1.127158,1.127158,Group 69,Group 70,Group 70,NaN,1.127158,1.127158,1.127158,1.127158
355334,358522,923.0,CBOS_331_12_2017.sav,2017,12,41.0,1976.0,1.0,Mężczyzna,5.0,...,1.716070,1.716070,Group 69,Group 47,Group 48,NaN,1.716070,1.716070,1.716070,1.716070
355335,358523,924.0,CBOS_331_12_2017.sav,2017,12,46.0,1971.0,2.0,Kobieta,5.0,...,1.572886,1.572886,Group 69,Group 47,Group 48,NaN,1.572886,1.572886,1.572886,1.572886


In [4]:
# ============================================================
# Load CT JSON metadata (for inspection / debugging)
# ============================================================
import json

old_voiv_path        = JSON_ROOT / "CBOS_old_voiv"        / "CBOS_all_years.json"
new_voiv_path        = JSON_ROOT / "CBOS_new_voiv"        / "CBOS_all_years.json"
old_groups_path      = JSON_ROOT / "CBOS_U_old_voiv_dict" / "CBOS_all_groups.json"
new_groups_path      = JSON_ROOT / "CBOS_U_new_voiv_dict" / "CBOS_all_groups.json"

with open(old_voiv_path,   encoding='utf-8') as f: ct_old_voiv   = json.load(f)
with open(new_voiv_path,   encoding='utf-8') as f: ct_new_voiv   = json.load(f)
with open(old_groups_path, encoding='utf-8') as f: ct_old_groups = json.load(f)
with open(new_groups_path, encoding='utf-8') as f: ct_new_groups = json.load(f)

print("Old voiv years:", list(ct_old_voiv.keys()))
print("New voiv years:", list(ct_new_voiv.keys()))
print(f"Old groups: {len(ct_old_groups)} groups, "
      f"years {sorted(set(y for g in ct_old_groups.values() for y in g))[:3]}…")
print(f"New groups: {len(ct_new_groups)} groups, "
      f"years {sorted(set(y for g in ct_new_groups.values() for y in g))[:3]}…")

# Quick sanity: keys in one CT record
example = ct_new_voiv['2005']['małopolskie']['2005']
print("\nExample CT keys (małopolskie 2005):", list(example.keys()))
print("  pop_class:", example['pop_class'])


Old voiv years: ['1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000']
New voiv years: ['1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017']
Old groups: 125 groups, years ['1990', '1991', '1992']…
New groups: 291 groups, years ['1999', '2000', '2001']…

Example CT keys (małopolskie 2005): ['E_age_sex_2000', 'E_educ_sex_2000', 'E_hh_size_2000', 'pop_class']
  pop_class: {'miasto od 20 001 do 50 000': 283343.0, 'wieś': 1551144.0, 'miasto do 20 000': 369793.0, 'miasto 500 001 i więcej': 756629.0, 'miasto od 50 001 do 100 000': 84729.0, 'miasto od 100 001 do 500 000': 117560.0}


In [5]:
# ============================================================
# Run raking for all weight columns (individual + household)
# ============================================================
from regional_weighting import compute_regional_weights

df = compute_regional_weights(
    df,
    json_root=JSON_ROOT,
    max_iter=100,
    tol=1e-6,
    verbose=True,
)

print("\nDone. Weight column summaries:")
weight_cols = [
    'weight_VOIV', 'weight_VOIV_h',
    'weight_VOIV_500', 'weight_VOIV_500_h',
    'weight_VOIV_100_500', 'weight_VOIV_100_500_h',
    'weight_VOIV_100', 'weight_VOIV_100_h',
]
for col in weight_cols:
    s = df[col]
    print(f"  {col:<28}: mean={s.mean():.4f}  min={s.min():.4f}  "
          f"max={s.max():.4f}  NaN={s.isna().sum()}")


Loading CT JSON files …

=== weight_VOIV / weight_VOIV_h ===


  Ind 1990: done  (mean=2472.8870, min=0.0000, max=123027.3795)


  Ind 1991: done  (mean=2983.7517, min=0.0000, max=119472.5181)


  Ind 1992: done  (mean=4332.7727, min=0.0000, max=111619.2099)


  Ind 1993: done  (mean=2840.6727, min=0.0000, max=88833.9840)


...

  HH 2013: done  (mean=1156.2638, min=0.0000, max=13733.2446)


  HH 2014: done  (mean=1103.2055, min=0.0000, max=13303.3351)


  HH 2015: done  (mean=1067.3008, min=0.0000, max=15308.7945)


  HH 2016: done  (mean=1047.8679, min=0.0000, max=20377.0154)


  HH 2017: done  (mean=1454.6105, min=0.0000, max=94339.0530)

Done. Weight column summaries:
  weight_VOIV                 : mean=3121.1794  min=0.0000  max=153590.5717  NaN=0
  weight_VOIV_h               : mean=1041.5443  min=0.0000  max=56193.6723  NaN=0
  weight_VOIV_500             : mean=3133.2824  min=0.0000  max=191314.9000  NaN=0
  weight_VOIV_500_h           : mean=1052.7872  min=0.0000  max=86789.3297  NaN=0
  weight_VOIV_100_500         : mean=3027.0942  min=0.0000  max=191314.9000  NaN=0
  weight_VOIV_100_500_h       : mean=1060.0169  min=0.0000  max=118249.3600  NaN=0
  weight_VOIV_100             : mean=3065.4834  min=0.0000  max=179732.7300  NaN=0
  weight_VOIV_100_h           : mean=1054.0154  min=0.0000  max=94339.0530  NaN=0


In [6]:
# ============================================================
# Validation: check that weighted marginals match CT targets
# for a sample group + year
# ============================================================

def validate_group(df, year, voiv_name, is_old=True):
    """
    Print weighted marginals vs CT targets for one voivodeship × year.
    Uses weight_VOIV (individual) and weight_VOIV_h (household).
    """
    src = ct_old_voiv if is_old else ct_new_voiv
    loc_col   = 'location_old_L' if is_old else 'location_new_L'
    age_col   = 'age_1990_L'     if is_old else 'age_2000_L'
    educ_col  = 'educ_1990_L'    if is_old else 'educ_2000_L'
    hh_col    = 'hh_size_1990_L' if is_old else 'hh_size_2000_L'
    age_key   = 'E_age_sex_1990' if is_old else 'E_age_sex_2000'
    educ_key  = 'E_educ_sex_1990' if is_old else 'E_educ_sex_2000'
    hh_key    = 'E_hh_size_1990' if is_old else 'E_hh_size_2000'

    from regional_weighting import EDUC_1990_REMAP

    try:
        ct = src[str(year)][voiv_name][str(year)]
    except KeyError:
        print(f"No CT for {voiv_name} {year}"); return

    sub = df[(df['survey_year'] == year) & (df[loc_col] == voiv_name)].copy()
    if len(sub) == 0:
        print(f"No observations for {voiv_name} {year}"); return

    print(f"\n{'='*60}")
    print(f"Voivodeship: {voiv_name}  Year: {year}  N={len(sub)}")
    print(f"{'='*60}")

    # --- age × sex (individuals) ---
    print("\n[Individual] age × sex — weighted vs target:")
    age_sex_ct = ct.get(age_key, {})
    for key in sorted(age_sex_ct):
        if 'ogółem' in key: continue
        age_lbl, sex_json = [s.strip() for s in key.split('×', 1)]
        sex_df = 'Kobieta' if sex_json == 'kobiety' else 'Mężczyzna'
        mask = (sub[age_col] == age_lbl) & (sub['sex_L'] == sex_df)
        w_sum = (sub.loc[mask, 'weight_VOIV']).sum()
        target = age_sex_ct[key]
        print(f"  {key:<45} weighted={w_sum:>12.1f}  target={target:>12.1f}  "
              f"ratio={w_sum/target:.4f}" if target > 0 else f"  {key}: target=0")

    # --- educ × sex (individuals) ---
    print("\n[Individual] educ × sex — weighted vs target:")
    educ_sex_ct = ct.get(educ_key, {})
    educ_s = sub[educ_col].map(lambda x: EDUC_1990_REMAP.get(x, x)) if is_old else sub[educ_col]
    for key in sorted(educ_sex_ct):
        if 'ogółem' in key: continue
        educ_lbl, sex_json = [s.strip() for s in key.split('×', 1)]
        sex_df = 'Kobieta' if sex_json == 'kobiety' else 'Mężczyzna'
        mask = (educ_s == educ_lbl) & (sub['sex_L'] == sex_df)
        w_sum = (sub.loc[mask, 'weight_VOIV']).sum()
        target = educ_sex_ct[key]
        print(f"  {key:<55} weighted={w_sum:>12.1f}  target={target:>12.1f}  "
              f"ratio={w_sum/target:.4f}" if target > 0 else f"  {key}: target=0")

    # --- hh_size (household weights) ---
    print("\n[Household] hh_size — weighted vs target:")
    hh_ct = ct.get(hh_key, {})
    for hh_lbl, target in hh_ct.items():
        if hh_lbl == 'ogółem': continue
        mask = sub[hh_col] == hh_lbl
        w_sum = sub.loc[mask, 'weight_VOIV_h'].sum()
        print(f"  {hh_lbl:<25} weighted={w_sum:>12.1f}  target={target:>12.1f}  "
              f"ratio={w_sum/target:.4f}" if target > 0 else f"  {hh_lbl}: target=0")

    # --- pop_class ---
    from regional_weighting import CS_PROB_TO_POP_CLASS
    print("\n[Individual] pop_class — weighted vs target:")
    pc_ct = ct.get('pop_class', {})
    for cs_val in sorted(sub['cs_prob'].dropna().unique()):
        keys = CS_PROB_TO_POP_CLASS.get(float(cs_val), [])
        target = sum(pc_ct.get(k, 0.0) for k in keys)
        mask = sub['cs_prob'] == cs_val
        w_sum = sub.loc[mask, 'weight_VOIV'].sum()
        label = '+'.join(keys) if keys else str(cs_val)
        print(f"  cs_prob={cs_val}  {label[:40]:<40} weighted={w_sum:>12.1f}  "
              f"target={target:>12.1f}"
              + (f"  ratio={w_sum/target:.4f}" if target > 0 else "  target=0"))

# Run validation for one old-voiv year and one new-voiv year
validate_group(df, year=1995, voiv_name='bydgoskie',  is_old=True)
validate_group(df, year=2005, voiv_name='małopolskie', is_old=False)



Voivodeship: bydgoskie  Year: 1995  N=463

[Individual] age × sex — weighted vs target:
  0-9 × kobiety                                 weighted=         0.0  target=     79214.0  ratio=0.0000
  0-9 × mężczyźni                               weighted=         0.0  target=     82942.0  ratio=0.0000
  10-19 × kobiety                               weighted=    156419.8  target=     98174.0  ratio=1.5933
  10-19 × mężczyźni                             weighted=    163329.4  target=    102550.0  ratio=1.5927
  20-29 × kobiety                               weighted=    126867.5  target=     79626.0  ratio=1.5933
  20-29 × mężczyźni                             weighted=    129844.9  target=     81526.0  ratio=1.5927
  30-39 × kobiety                               weighted=    137606.3  target=     86366.0  ratio=1.5933
  30-39 × mężczyźni                             weighted=    137728.7  target=     86476.0  ratio=1.5927
  40-49 × kobiety                               weighted=    146522.3  

In [7]:
# ============================================================
# Save results
# ============================================================
output_path = REWEIGHTING_ROOT / 'CBOS_data_reweighted.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved {df.shape[0]:,} rows × {df.shape[1]} cols → {output_path}")


Saved 355,337 rows × 96 cols → /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/reweighting/CBOS_data_reweighted.csv
